## 1. Setup

Load libraries and the model table built in `00_build_model_table.ipynb`.
Each row is one session that added at least one item to the cart; `label = 1` means the session purchased after its first add-to-cart.
The data is split by time: sessions before Oct 25, 2019 are used for training, the rest for testing.

### Data dictionary: `model_table`

| Column | Type | Description |
|---|---|---|
| `user_session` | string | Session ID (one row per session) |
| `user_id` | int | User ID |
| `first_cart_time` | timestamp | Time of the first add-to-cart in the session |
| `label` | int | 1 if the session purchased after its first add-to-cart, else 0 |
| `n_views_before_cart` | int | Number of product views before the first add-to-cart |
| `n_products_viewed_before_cart` | int | Number of distinct products viewed before the first add-to-cart |
| `secs_to_first_cart` | int | Seconds from session start to the first add-to-cart |
| `cart_hour` | int | Hour of the first add-to-cart (UTC) |
| `cart_dow` | int | Day of week of the first add-to-cart (0 = Sunday) |
| `cart_product_id` | int | ID of the carted product (not used as a feature) |
| `cart_price` | float | Price of the carted product |
| `cart_category_id` | int | Category ID of the carted product (categorical) |
| `cart_brand_missing` | int | 1 if the carted product has no brand recorded |
| `split` | string | `train` (before Oct 25, 2019) or `test` |

User-history features (`prior_purchases`, `prior_carts`, `prior_sessions`) are added in Section 4.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import lightgbm as lgb

df = pd.read_parquet('../data/processed/model_table.parquet')
df = df.sort_values('user_session').reset_index(drop=True)   # fix row order for reproducibility

train = df[df['split'] == 'train']
test = df[df['split'] == 'test']
y_train = train['label']
y_test = test['label']

num_features = [
    'n_views_before_cart', 'n_products_viewed_before_cart', 'secs_to_first_cart',
    'cart_hour', 'cart_dow', 'cart_price', 'cart_brand_missing'
]

print(train.shape, test.shape)

(463222, 14) (109875, 14)


## 2. Baseline models

### Evaluation helper

All models are evaluated on the test set with three metrics:
- **ROC-AUC**: probability that a random purchaser is ranked above a random non-purchaser (0.5 = random, 1 = perfect)
- **PR-AUC**: average precision across thresholds; focuses on how well positive predictions hold up
- **Accuracy**: share of correct predictions at a 0.5 threshold; compare against the 51.98% baseline of predicting "purchase" for everyone

In [2]:
def evaluate(name, y_true, y_prob):
    return {
        'model': name,
        'roc_auc': round(roc_auc_score(y_true, y_prob), 4),
        'pr_auc': round(average_precision_score(y_true, y_prob), 4),
        'accuracy': round(accuracy_score(y_true, (y_prob >= 0.5).astype(int)), 4),
    }

results = []

### Logistic regression baseline

In [3]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(train[num_features], y_train)

p_lr = lr.predict_proba(test[num_features])[:, 1]
results.append(evaluate('Logistic regression', y_test, p_lr))
pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376


### LightGBM

Gradient-boosted decision trees: many small trees are built in sequence, each one correcting the errors of the previous ones. Unlike logistic regression, it captures non-linear relationships and interactions between features, and does not require standardization.

`cart_category_id` is added as a categorical feature. Category levels are taken from the training set only; categories that appear only in the test set are treated as missing.

Hyperparameters are reasonable defaults and were not tuned. The random seed and deterministic settings make results reproducible.

In [4]:
cats = train['cart_category_id'].astype('category').cat.categories

def prep(d):
    X = d[num_features].copy()
    values = d['cart_category_id'].where(d['cart_category_id'].isin(cats))   # unseen categories -> NaN
    X['cart_category_id'] = pd.Categorical(values, categories=cats)
    return X

X_train, X_test = prep(train), prep(test)

model = lgb.LGBMClassifier(
    n_estimators=300,       # number of trees
    learning_rate=0.05,     # how much each tree contributes
    num_leaves=31,          # max leaves per tree (controls tree complexity)
    random_state=42,        # fixed seed for reproducibility
    deterministic=True,
    force_row_wise=True,
    verbose=-1,             # suppress training logs
)
model.fit(X_train, y_train)

p_lgb = model.predict_proba(X_test)[:, 1]
results.append(evaluate('LightGBM', y_test, p_lgb))
pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6250,0.6343,0.5882


**Interpretation**

- **Logistic regression:** ROC-AUC 0.545, barely above random (0.5). Accuracy 53.8%, only 1.8 percentage points above the 51.98% baseline.
- **LightGBM:** ROC-AUC 0.625, accuracy 58.8%, about 6.8 percentage points above the baseline.

LightGBM is clearly better, but 0.625 is still modest predictive power. The current features only describe pre-cart behavior within the session and the carted product.

The gap between the two models has two possible causes, which cannot be separated yet:
1. LightGBM captures non-linear relationships; logistic regression does not. Long-tailed features such as `secs_to_first_cart` are also hard for logistic regression without a log transform.
2. LightGBM uses `cart_category_id`; logistic regression does not.

The next section separates these two effects.

## 3. Ablation & importance

### Feature importance

Gain importance: the total improvement in the training loss contributed by all splits on a feature. Higher means the model relied on it more during training.

In [5]:
# Gain importance from the full LightGBM model
importance = pd.DataFrame({
    'feature': X_train.columns,
    'gain': model.booster_.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)

# Share of total gain, in percent
importance['gain_pct'] = (100 * importance['gain'] / importance['gain'].sum()).round(1)
importance

,feature,gain,gain_pct
7,cart_category_id,121248.385406,37.1
2,secs_to_first_cart,120882.010843,37.0
5,cart_price,32086.338786,9.8
3,cart_hour,25045.378819,7.7
0,n_views_before_cart,11085.884985,3.4
4,cart_dow,8450.019032,2.6
1,n_products_viewed_before_cart,7728.333912,2.4
6,cart_brand_missing,28.795530,0.0


**Interpretation**

- `cart_category_id` (37.1%) and `secs_to_first_cart` (37.0%) are essentially tied at the top.
- `secs_to_first_cart`: how long it takes a user to add to cart is related to whether they buy. The relationship is likely non-linear (very fast and very slow carts may reflect different intent), which is worth exploring in a follow-up EDA.
- `cart_category_id`: gain importance tends to overstate high-cardinality categorical features. The ablation below tests whether it actually helps on the test set.
- `cart_brand_missing` has near-zero gain: a missing brand carries no predictive signal.

### LightGBM without `cart_category_id`

Same model and hyperparameters, using only the 7 numeric features that logistic regression uses. This isolates the effect of the model type from the effect of the category feature.

In [6]:
# Same LightGBM settings, numeric features only (no category)
model_nocat = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
model_nocat.fit(train[num_features], y_train)

p_nocat = model_nocat.predict_proba(test[num_features])[:, 1]
results.append(evaluate('LightGBM (no category)', y_test, p_nocat))
pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6250,0.6343,0.5882
2,LightGBM (no category),0.6172,0.6287,0.5801


**Interpretation**

- **LightGBM vs. LightGBM (no category):** ROC-AUC drops only from 0.625 to 0.617 (−0.008). The category feature contributes little.
- **LightGBM (no category) vs. logistic regression:** 0.617 vs. 0.545 with the exact same features. The model type accounts for most of the gap.

**Conclusion:** LightGBM outperforms logistic regression mainly because it captures non-linear relationships, not because of the category feature.

This also shows that gain importance and actual contribution can disagree: `cart_category_id` ranks first by gain but barely changes test performance when removed. Ablation is the more reliable check.

## 4. User history features

Adds three features describing the user's behavior in **earlier sessions**, before the current first add-to-cart (built in `00_build_model_table.ipynb`):
- `prior_purchases`: number of purchases
- `prior_carts`: number of add-to-cart events
- `prior_sessions`: number of sessions

Both models are retrained with the original features plus these three, using the same split and hyperparameters.

In [7]:
# Load user-history features and join them to the model table by session
hist = pd.read_parquet('../data/processed/feat_user_history.parquet')
df2 = df.merge(hist, on='user_session', how='left')

train2 = df2[df2['split'] == 'train']
test2 = df2[df2['split'] == 'test']
y_train2 = train2['label']
y_test2 = test2['label']

# Original numeric features + three user-history features
num_features_v2 = num_features + ['prior_purchases', 'prior_carts', 'prior_sessions']

print(train2.shape, test2.shape)

(463222, 17) (109875, 17)


In [8]:
# Logistic regression with history features
lr2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr2.fit(train2[num_features_v2], y_train2)

p_lr2 = lr2.predict_proba(test2[num_features_v2])[:, 1]
results.append(evaluate('Logistic regression + history', y_test2, p_lr2))

# LightGBM with history features and cart_category_id
def prep_v2(d):
    X = d[num_features_v2].copy()
    values = d['cart_category_id'].where(d['cart_category_id'].isin(cats))   # unseen categories -> NaN
    X['cart_category_id'] = pd.Categorical(values, categories=cats)
    return X

X_train2, X_test2 = prep_v2(train2), prep_v2(test2)

model2 = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
model2.fit(X_train2, y_train2)

p_lgb2 = model2.predict_proba(X_test2)[:, 1]
results.append(evaluate('LightGBM + history', y_test2, p_lgb2))

pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6250,0.6343,0.5882
2,LightGBM (no category),0.6172,0.6287,0.5801
3,Logistic regression + history,0.6955,0.7250,0.6396
4,LightGBM + history,0.7296,0.7508,0.6674


**Interpretation**

| Model | Before | With history | Change |
|---|---|---|---|
| Logistic regression | 0.545 | 0.696 | +0.151 |
| LightGBM | 0.625 | 0.730 | +0.105 |

- **Who the user is matters a lot, not just what they do in the session.** For logistic regression, three history features improved ROC-AUC far more than all session features combined (+0.151 vs. +0.045 over random). For LightGBM, they added +0.105, close to what the first eight features achieved over random (+0.125). LightGBM + history reaches 66.7% accuracy, 14.8 percentage points above the baseline.
- **Logistic regression improved more**, narrowing the gap to LightGBM from 0.08 to 0.03. The relationship between history and purchase is roughly monotonic (users with more past purchases are more likely to buy), which a linear model captures well.
- **Caveat (left censoring):** the data starts on Oct 1, so late-October test sessions have more visible history than early-October training sessions. Feature distributions differ between train and test, which may make test results slightly optimistic. Adding September data would help verify this.

## 5. Targeting analysis

How well does the model find carts that will be abandoned? Test sessions are ranked by predicted abandonment probability (LightGBM + history), and the actual abandonment rate is measured within the top k%.

- **abandon_rate**: share of sessions in the top k% that actually abandoned
- **lift**: abandon_rate divided by the overall abandonment rate, i.e., how many times better than picking sessions at random. The 100% row covers all sessions, so its lift is always 1.0

In [9]:
# Probability of abandonment = 1 - probability of purchase
p_abandon = 1 - p_lgb2
abandoned = 1 - y_test2.values
base_rate = abandoned.mean()          # overall abandonment rate in the test set

# Session indices sorted from highest to lowest abandonment probability
# (argsort sorts ascending, so the probabilities are negated)
order = np.argsort(-p_abandon)

rows = []
for k in [0.1, 0.2, 0.3, 0.5, 1.0]:
    n = int(len(order) * k)                    # number of sessions in the top k%
    rate = abandoned[order[:n]].mean()         # actual abandonment rate among them
    rows.append({
        'top_pct': int(k * 100),
        'n_sessions': n,
        'abandon_rate': round(rate, 4),
        'lift': round(rate / base_rate, 2),
    })

pd.DataFrame(rows)

,top_pct,n_sessions,abandon_rate,lift
0,10,10987,0.7605,1.58
1,20,21975,0.7227,1.51
2,30,32962,0.6967,1.45
3,50,54937,0.6478,1.35
4,100,109875,0.4802,1.00


**Interpretation**

The overall abandonment rate in the test set is 48.0%. Among the carts the model ranks as riskiest:

- **Top 10%:** 76.1% actually abandon (lift 1.58)
- **Top 20%:** 72.3% actually abandon (lift 1.51)

Targeting the top 20% with reminders would reach real abandoners about 72% of the time, versus 48% with random targeting. A wider cutoff reaches more abandoners but with lower precision; the right cutoff depends on reminder cost and budget.

**Caveat:** the model identifies who is likely to abandon, not who will respond to a reminder. Some abandoners will not buy regardless. Measuring reminder effectiveness requires an A/B test, a natural next step toward uplift modeling.